# Humanitas Purissima: Orakel KL-Divergenz Simulation
Dieses Notebook simuliert den **Kullback-Leibler Entropie-Decay** zur Abwehr von Hardware-Spoofing gemäß Artikel II.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy

def simulate_oracle_network(num_steps=50, alpha=0.8, w_threshold=0.2):
    np.random.seed(42)
    bins = np.linspace(0, 100, 20)
    true_mean, true_std = 50.0, 10.0
    num_sensors = 10
    weights = np.ones(num_sensors)
    weight_history = [weights.copy()]
    spoofed_sensor_id = 0
    attack_step = 10
    
    for t in range(num_steps):
        sensor_data = []
        for i in range(num_sensors):
            if i == spoofed_sensor_id and t >= attack_step:
                data = np.random.normal(75.0, 8.0, 500)
            else:
                data = np.random.normal(true_mean + np.random.uniform(-1, 1), true_std, 500)
            hist, _ = np.histogram(data, bins=bins, density=True)
            hist = np.where(hist == 0, 1e-6, hist)
            hist /= np.sum(hist)
            sensor_data.append(hist)
            
        P_kontext = np.average(sensor_data, axis=0, weights=weights)
        P_kontext /= np.sum(P_kontext)
        
        for i in range(num_sensors):
            P_sensor = sensor_data[i]
            d_kl = entropy(P_sensor, P_kontext)
            weights[i] = weights[i] * np.exp(-alpha * d_kl)
            
        weight_history.append(weights.copy())

    weight_history = np.array(weight_history)
    plt.figure(figsize=(10, 5))
    for i in range(num_sensors):
        if i == spoofed_sensor_id:
            plt.plot(weight_history[:, i], label=f'Sensor {i} (Spoofed)', color='red', linewidth=2.5)
        else:
            plt.plot(weight_history[:, i], color='gray', alpha=0.5)
    plt.axhline(y=w_threshold, color='black', linestyle='--', label=f'Threshold ({w_threshold})')
    plt.title('KL-Divergenz Entropie-Decay (Artikel II)')
    plt.xlabel('Schritte (t)')
    plt.ylabel('Gewichtung (w_i)')
    plt.legend()
    plt.grid(True)
    plt.show()

simulate_oracle_network()